In [3]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [4]:
import os
from docent.data_models.agent_run import AgentRun
from docent._loader.load_inspect import load_inspect_eval
os.environ['ENV_TYPE'] = 'swe-bench'
# from docent._loader.load_inspect import load_swebench

SWE_BENCH_LOGS: dict[str, str | tuple[str, dict[str, str]]] = {
    # "swebench-sonnet37-old": f"/home/ubuntu/artifacts/vincent/swe_bench_logs/2025-04-09T21-09-59+00-00_swe-bench_8AcW4AHxbhgtoqEbe5FQcT.eval",
    # "swebench-sonnet37-new": f"/home/ubuntu/artifacts/vincent/swe_bench_logs/2025-04-10T21-39-15+00-00_swe-bench_TZrCQjagGBxzuSrXnE3fqj.eval",
    "swebench-sonnet35-tools": "~/projects/docent-artifacts/eval1.eval",
    "swebench-sonnet37-tools": "~/projects/docent-artifacts/eval2.eval",
}
def load_swebench() -> list[AgentRun]:
    return load_inspect_eval(SWE_BENCH_LOGS)

TRANSCRIPTS = load_swebench()

20:06:20 [INFO] docent._loader.load_inspect: Loading swebench-sonnet35-tools from ~/projects/docent-artifacts/eval1.eval
20:06:21 [INFO] docent._loader.load_inspect: Loading swebench-sonnet37-tools from ~/projects/docent-artifacts/eval2.eval


In [5]:
from docent.data_models.agent_run import AgentRun

def find_by_expid(transcripts: list[AgentRun], expid: str) -> AgentRun:
    for t in transcripts:
        if t.metadata.get("experiment_id") == expid:
            return t
    raise ValueError(f"No transcript found for experiment ID {expid}")

def printw(s: str):
    LINE_LENGTH = 150
    lines = s.split('\n')
    for line in lines:
        for i in range(0, len(line), LINE_LENGTH):
            print(line[i:i+LINE_LENGTH])

In [6]:
from docent.data_models.transcript import (
    MULTI_BLOCK_CITE_INSTRUCTION,
)
from docent._llm_util.prod_llms import get_llm_completions_async
from docent._llm_util.providers.preferences import PROVIDER_PREFERENCES

# Multi-Stage

### Run it

In [8]:
import os
from docent.data_models.agent_run import AgentRun


os.environ['ENV_TYPE'] = 'swe-bench'



In [10]:

all_sample_ids: set[str] = {
    "astropy__astropy-14096", "django__django-11551", "django__django-11239", "django__django-14351",
    "django__django-11299", "django__django-11749", "django__django-12193", "django__django-14404",
}
print(TRANSCRIPTS[0:4])
TRANSCRIPT_PAIRS: dict[str, list[AgentRun]] = {}
for sample_id in all_sample_ids:
    sample_transcripts = [t for t in TRANSCRIPTS if t.metadata.get("sample_id") == sample_id]
    if len(sample_transcripts) != 2:
        continue
    first, second = find_by_expid(sample_transcripts, 'swebench-sonnet37-tools'), find_by_expid(sample_transcripts, 'swebench-sonnet35-tools')
    TRANSCRIPT_PAIRS[sample_id] = [first, second]

len(TRANSCRIPT_PAIRS)

len(TRANSCRIPT_PAIRS)
msgs = next(iter(TRANSCRIPT_PAIRS.items()))[1][0].transcripts['default'].messages

for i in range(7):
    print("~~{}~ ~{}~~\n{}\n\n".format(i, msgs[i].role, msgs[i].text))


[AgentRun(id='14556c76-43b7-427a-91b3-80fb00d076c4', name=None, description=None, transcripts={'default': Transcript(id='58639f6f-c6db-419a-b571-e9362f5dce6e', name=None, description=None, messages=[SystemMessage(id='D9aWuy55AnggHYGpZet2Qw', content="\nI've uploaded a python code repository in a directory. In the next message, you will be given a PR description for a coding task that you should solve.\n\nPlease implement the necessary changes to the repository so that the requirements specified in the description are met\nI've already taken care of all changes to any of the test files described in the description. This means you DON'T have to modify the testing logic or any of the tests in any way!\n\nYour task is to make the minimal changes to non-tests files in the existing repository to ensure the PR description is satisfied.\n\nFollow these steps to resolve the issue:\n1. As a first step, it might be a good idea to explore the repo to familiarize yourself with its structure.\n2. Cr

In [12]:
pair = next(iter(TRANSCRIPT_PAIRS.items()))[1]
from docent._ai_tools.diffs.llm_diff_summaries import compute_transcript_diff
x =  await compute_transcript_diff(pair[0], pair[1])
print(x)


# swe_diffs = await extract_states_and_diffs()
# swe_diffs.keys()


20:14:09 [INFO] docent._llm_util.prod_llms: claude-3-7-sonnet-20250219: 1 cache hits, 0 misses
20:14:09 [INFO] docent._llm_util.prod_llms: claude-3-7-sonnet-20250219: 1 cache hits, 0 misses
20:14:09 [INFO] docent._llm_util.prod_llms: claude-3-7-sonnet-20250219: 1 cache hits, 0 misses
agent_run_1_id='c3e2962f-e2f3-4f87-bbf3-041978b41e8d' agent_run_2_id='6f1e2ee1-cbaf-4a8b-9791-4c52c0167166' claims=[Claim(claim_summary='Agent 1 jumps directly to examining specific files while Agent 2 systematically explores the repository structure first\n    \n    <agent_1_action>\n        Immediately begins examining specific Django files like constraints.py without showing systematic repository exploration\n    </agent_1_action>\n    <agent_2_action>\n        Uses bash commands to systematically explore the repository structure before diving into specific files\n    </agent_2_action>\n    <evidence>\n        Agent 1 directly examines specific files [T0B4][T0B6][T0B8]\n        Agent 2 systematically ex

In [14]:
from docent._ai_tools.diffs.llm_message_summaries import get_llm_output_for_transcript_to_message_summaries
text = await get_llm_output_for_transcript_to_message_summaries(pair[0])

20:14:20 [INFO] docent._llm_util.prod_llms: claude-3-7-sonnet-20250219: 1 cache hits, 0 misses


In [15]:
from docent._ai_tools.diffs.llm_message_summaries import llm_output_to_message_summaries
messages = llm_output_to_message_summaries(text)

### Vincent's original diffing (evidence + claim)

In [17]:
from docent._ai_tools.diffs.llm_diff_summaries import compare_transcript_states

messages_0 = llm_output_to_message_summaries(await get_llm_output_for_transcript_to_message_summaries(pair[0]))
messages_1 = llm_output_to_message_summaries(await get_llm_output_for_transcript_to_message_summaries(pair[1]))

diff_text = await compare_transcript_states(pair[0], pair[1], messages_0, messages_1)
print(diff_text)

20:14:48 [INFO] docent._llm_util.prod_llms: claude-3-7-sonnet-20250219: 1 cache hits, 0 misses
20:14:48 [INFO] docent._llm_util.prod_llms: claude-3-7-sonnet-20250219: 1 cache hits, 0 misses
20:14:48 [INFO] docent._llm_util.prod_llms: claude-3-7-sonnet-20250219: 1 cache hits, 0 misses
<claim>
Both agents aim to understand the codebase to diagnose the CheckConstraint issue, but Agent 1 immediately jumps to examining specific files while Agent 2 uses systematic bash commands to explore the repository structure first.
</claim>
<evidence>
At the beginning of their analysis, both agents want to understand where the CheckConstraint SQL generation issue occurs. Agent 1 immediately starts by saying they'll analyze the issue and directly examines the constraints.py file [T0B2-T0B4]. In contrast, Agent 2 runs bash commands to systematically explore the repository structure first [T1B2], and only after getting an overview does it narrow down to relevant files [T1B4] before examining constraints.py

### My approach

In [19]:
from docent._ai_tools.diffs.llm_diff_summaries import get_llm_output_compare_transcript_states_v2

messages_0 = llm_output_to_message_summaries(await get_llm_output_for_transcript_to_message_summaries(pair[0]))
messages_1 = llm_output_to_message_summaries(await get_llm_output_for_transcript_to_message_summaries(pair[1]))

diff_text = await get_llm_output_compare_transcript_states_v2(pair[0], pair[1], messages_0, messages_1)
print(diff_text)

20:15:37 [INFO] docent._llm_util.prod_llms: claude-3-7-sonnet-20250219: 1 cache hits, 0 misses
20:15:37 [INFO] docent._llm_util.prod_llms: claude-3-7-sonnet-20250219: 1 cache hits, 0 misses
20:15:37 [INFO] docent._llm_util.prod_llms: claude-3-7-sonnet-20250219: 1 cache hits, 0 misses
<claim>
    Agent 1 jumps directly to examining specific files while Agent 2 systematically explores the repository structure first
    <shared_context>
        Both agents are starting their work and need to understand the codebase structure to locate the issue with CheckConstraint SQL generation
    </shared_context>
    <agent_1_action>
        Immediately begins examining specific Django files like constraints.py without showing systematic repository exploration
    </agent_1_action>
    <agent_2_action>
        Uses bash commands to systematically explore the repository structure before diving into specific files
    </agent_2_action>
    <evidence>
        Agent 1 directly examines specific files [T0

In [20]:
pair

[AgentRun(id='c3e2962f-e2f3-4f87-bbf3-041978b41e8d', name=None, description=None, transcripts={'default': Transcript(id='7963df1d-1036-4a5c-90ab-6c399512cb94', name=None, description=None, messages=[SystemMessage(id='2NNmp3y5gwDdPt34htcowE', content="\nI've uploaded a python code repository in a directory. In the next message, you will be given a PR description for a coding task that you should solve.\n\nPlease implement the necessary changes to the repository so that the requirements specified in the description are met\nI've already taken care of all changes to any of the test files described in the description. This means you DON'T have to modify the testing logic or any of the tests in any way!\n\nYour task is to make the minimal changes to non-tests files in the existing repository to ensure the PR description is satisfied.\n\nFollow these steps to resolve the issue:\n1. As a first step, it might be a good idea to explore the repo to familiarize yourself with its structure.\n2. Cr

### Generate the diff

In [29]:
from docent._ai_tools.diffs.llm_diff_summaries import compute_transcript_diff, _parse_llm_output_to_claims
diff_text = await get_llm_output_compare_transcript_states_v2(pair[0], pair[1], messages_0, messages_1)
parsed_claims = _parse_llm_output_to_claims(diff_text)
[pc.model_dump() for pc in parsed_claims]
# diff = await compute_transcript_diff(pair[0], pair[1])
# [print(c.claim_summary)for c in diff.claims] 

20:19:13 [INFO] docent._llm_util.prod_llms: claude-3-7-sonnet-20250219: 1 cache hits, 0 misses


AttributeError: 'list' object has no attribute 'claim_summary'